# Problem ID: Category10_Problem01 — Emergency Drone Dispatch Contract Auction

**Faculty:** Dheeraj Pandey  |  **Course:** CS205B — Artificial Intelligence and Its Application  
**Assigned category:** 10 — Multi-Agent Resource Bidding  |  **Syllabus:** Unit 2 (intelligent agents, utility-based agents, MAS coordination and communication); Unit 4 (cooperative/competitive agents, applied multi-agent simulation).

### 1. Problem Scenario
After a flood, a command centre broadcasts emergency medicine delivery contracts to independently operated drone companies. Drones reply with price and time bids. The command-centre manager must award each contract while respecting battery capacity, crosswind tolerance, delivery deadlines and budgets.

The simulated **Contract Net Protocol (CNP)** is: manager issues a *call for proposals* → drone companies independently produce bids → manager scores and awards a feasible bid → the environment reports completion quality → the manager updates its estimates. All data and the environment are supplied, so **no internet access** is needed.

### 2. System Specifications
- **State space $S$:** current medical deliveries request (`id`, `qty`, `distance`, `priority`, `budget`, `deadline`, quality constraint), eligible bid messages (`id`, `price`, `eta`, `quality`, `metric`, `capability`, `capacity`, and `role`), and the manager's trust and within-episode workload vectors.
- **Resource meaning:** `qty` consumes battery credits on each selected contractor after accepting a contract. Each contractor starts with finite capacity, so later requests see reduced availability.
- **Contract action:** an integer contractor ID (`0` to `len(CONTRACTORS)-1`); return `-1` if no offer is feasible.
- **Bidding mechanics:** Each supplier quotes **base fee + (per-unit price × demand) + (distance tariff × distance) + (priority surcharge × priority)**; ETA = setup time + distance/speed + demand/throughput. Offers are constructed in locked Cell 3.
- **Eligibility:** Sufficient resource capacity; `price <= budget`; `eta <= deadline`; the advertised **crosswind tolerance** must be **at least** `min_metric`; when `need_flag = 1`, the contractor must provide **refrigerated medical compartment**. Drones must have sufficient battery credits for the request and meet the wind-tolerance threshold; chilled deliveries need a refrigeration compartment.
- **Estimated quality and contractor trust:** quoted `quality` and learned `trust` lie in `[0,1]`; `metric` is the scenario-specific numerical compliance rating.
- **Reward/transition:** a completed, awarded request produces `priority × 14 − 0.12 × price − 0.70 × ETA`; an unsuccessful awarded request produces `priority × (−9) − 0.12 × price − 0.70 × ETA`; no admissible contract incurs `−6 × priority`. The awarded contractor(s) lose `qty` stock whether the attempt succeeds or fails. Episodes reset resource stock but keep the manager's learned trust.
- **Decision parameters:** `ALPHA` = feedback learning rate, `GAMMA` = trust weighting factor where present, `EPSILON` = exploration probability. All randomness is seeded at 42.

### 3. Student Implementation Deliverables
**Task 1 — Cell 4: CNP manager selection (`choose_action`).** Read the current request and independently submitted bids, apply **all** eligibility rules, and maximize this scenario's utility:

> **Utility =** priority × [9 × deadline slack + 5 × promised quality] + 14 × γ × learned trust − 0.32 × price − 4 × (1 − crosswind tolerance).

Use an $\epsilon$-exploration policy: with probability `EPSILON`, select a uniformly random feasible contract; otherwise use maximum utility. During exploration, pick a uniformly random feasible individual bidder. Break utility ties by the lowest numeric contractor ID. If no eligible contract exists, return `-1`. Do not edit the locked environment to bypass constraints.

**Task 2 — Cell 5: Learn from the awarded contract (`update_agent`).** For each selected contractor $i$, update its trust with

$$T_i \leftarrow (1-\alpha)T_i + \alpha \cdot y_i,$$

where **$y_i$ = 0.70 × success + 0.30 × on-time indicator**. Record each awarded contractor's `workload[i] += 1`. On a rejected/unassigned request, leave trust and workload unchanged. The `feedback` dictionary provides `awarded`, `selected`, `success`, `on_time`, `observed_quality`, and `individual_outcomes`. (The last field is especially important for the two-contractor task.)

**Verification — Cells 6 and 7 (locked):** Execute the entire notebook top-to-bottom; inspect total rewards, average awards/successes and the convergence/contract-outcome plots. The fixed driver includes basic assertions; an incorrect task implementation may fail during execution.

**Exam scope:** Complete only Cells 4 and 5. Cells 1–3 and 6–7 are locked. This is a medium-level offline pair-programming exercise targeted at approximately 1 hour 40 minutes.


In [ ]:
# Cell 2: Offline libraries, seeded scenario and parameters (LOCKED)
import numpy as np
import matplotlib.pyplot as plt
import random

random.seed(42)
np.random.seed(42)

PROBLEM_ID = "Category10_Problem01"
PROBLEM_TITLE = 'Emergency Drone Dispatch Contract Auction'
CONTRACTORS = [{'id': 'C1', 'capacity': 18, 'base_fee': 5.0, 'rate': 4.5, 'distance_fee': 0.8, 'speed': 6.0, 'throughput': 6.0, 'setup': 0.6, 'quality': 0.94, 'reliability': 0.87, 'metric': 0.9, 'capability': 1, 'role': 'contractor', 'workload': 0}, {'id': 'C2', 'capacity': 15, 'base_fee': 9.0, 'rate': 3.6, 'distance_fee': 0.5, 'speed': 4.5, 'throughput': 4.0, 'setup': 1.0, 'quality': 0.83, 'reliability': 0.8, 'metric': 0.82, 'capability': 0, 'role': 'contractor', 'workload': 0}, {'id': 'C3', 'capacity': 17, 'base_fee': 7.0, 'rate': 4.0, 'distance_fee': 1.0, 'speed': 5.0, 'throughput': 7.0, 'setup': 0.8, 'quality': 0.9, 'reliability': 0.92, 'metric': 0.72, 'capability': 1, 'role': 'contractor', 'workload': 0}, {'id': 'C4', 'capacity': 16, 'base_fee': 6.0, 'rate': 4.2, 'distance_fee': 0.7, 'speed': 7.5, 'throughput': 5.0, 'setup': 1.2, 'quality': 0.79, 'reliability': 0.77, 'metric': 0.95, 'capability': 1, 'role': 'contractor', 'workload': 0}, {'id': 'C5', 'capacity': 20, 'base_fee': 8.0, 'rate': 3.3, 'distance_fee': 0.9, 'speed': 6.0, 'throughput': 6.0, 'setup': 0.7, 'quality': 0.97, 'reliability': 0.84, 'metric': 0.67, 'capability': 0, 'role': 'contractor', 'workload': 0}]
REQUESTS = [{'id': 'R1', 'qty': 3, 'distance': 4, 'deadline': 6.0, 'budget': 55, 'priority': 3, 'min_metric': 0.65, 'need_flag': 0}, {'id': 'R2', 'qty': 5, 'distance': 7, 'deadline': 7.7, 'budget': 72, 'priority': 2, 'min_metric': 0.75, 'need_flag': 1}, {'id': 'R3', 'qty': 4, 'distance': 6, 'deadline': 6.5, 'budget': 65, 'priority': 3, 'min_metric': 0.82, 'need_flag': 0}, {'id': 'R4', 'qty': 6, 'distance': 9, 'deadline': 8.7, 'budget': 85, 'priority': 1, 'min_metric': 0.67, 'need_flag': 1}, {'id': 'R5', 'qty': 2, 'distance': 3, 'deadline': 5.7, 'budget': 49, 'priority': 2, 'min_metric': 0.85, 'need_flag': 0}, {'id': 'R6', 'qty': 4, 'distance': 8, 'deadline': 8.3, 'budget': 78, 'priority': 3, 'min_metric': 0.73, 'need_flag': 1}]
METRIC_DIRECTION = 'min'  # min => minimum threshold; max => upper bound
PRIORITY_SURCHARGE = 0.8
TEAM_MODE = False
HANDOFF_DELAY = 0.65
NUM_EPISODES = 40
MAX_STEPS_PER_EPISODE = len(REQUESTS)
ALPHA = 0.22  # trust update learning rate
GAMMA = 0.92  # trust-weight factor in applicable utility rules
EPSILON = 0.08  # exploration probability for CNP awards


In [ ]:
# Cell 3: Fully operational multi-agent environment engine (LOCKED)
class EnvironmentEngine:
    """Contract-Net simulation: call for proposals -> contractor bids -> award -> feedback."""

    def __init__(self):
        self.reset()

    def reset(self):
        self.contractors = [dict(c) for c in CONTRACTORS]
        self.pointer = 0
        self.history = []
        return self._state()

    def _propose(self, i, request):
        """Each independent contractor creates a bid using its own tariff and availability."""
        c = self.contractors[i]
        if c["capacity"] < request["qty"]:
            return None
        price = round(c["base_fee"] + c["rate"] * request["qty"]
                      + c["distance_fee"] * request["distance"]
                      + PRIORITY_SURCHARGE * request["priority"], 2)
        eta = round(c["setup"] + request["distance"] / c["speed"]
                    + request["qty"] / c["throughput"], 2)
        return {"id": i, "price": price, "eta": eta,
                "quality": c["quality"], "metric": c["metric"],
                "capability": c["capability"], "reliability": c["reliability"],
                "role": c["role"], "capacity": c["capacity"]}

    def _state(self):
        if self.pointer >= len(REQUESTS):
            return None
        req = dict(REQUESTS[self.pointer])
        bids = [self._propose(i, req) for i in range(len(self.contractors))]
        return {"request": req, "bids": [b for b in bids if b is not None],
                "round": self.pointer + 1}

    def _eligible(self, bid, req):
        if bid is None or bid["capacity"] < req["qty"]:
            return False
        if req["need_flag"] and not bid["capability"]:
            return False
        if METRIC_DIRECTION == "min" and bid["metric"] < req["min_metric"]:
            return False
        if METRIC_DIRECTION == "max" and bid["metric"] > req["min_metric"]:
            return False
        return True

    def step(self, action):
        """Award a contract; return next state, realized reward, terminal, feedback."""
        if self.pointer >= len(REQUESTS):
            raise RuntimeError("Call reset() before using step() at the end of an episode")
        state = self._state()
        req = state["request"]
        bids = {bid["id"]: bid for bid in state["bids"]}
        awarded, selected, price, eta = False, [], 0.0, 0.0
        success, on_time = False, False
        individual_outcomes = {}
        observed_quality = 0.0
        if TEAM_MODE:
            pair = tuple(action) if isinstance(action, (tuple, list)) else ()
            if len(pair) == 2 and pair[0] in bids and pair[1] in bids and pair[0] != pair[1]:
                pick, pack = bids[pair[0]], bids[pair[1]]
                price = round(pick["price"] + pack["price"], 2)
                eta = round(max(pick["eta"], pack["eta"]) + HANDOFF_DELAY, 2)
                admissible = (pick["role"] == "picker" and pack["role"] == "packer"
                              and self._eligible(pick, req) and self._eligible(pack, req)
                              and price <= req["budget"] and eta <= req["deadline"])
                if admissible:
                    selected = [pick["id"], pack["id"]]
                    awarded = True
                    observed_quality = min(pick["quality"], pack["quality"])
                    for b in (pick, pack):
                        individual_outcomes[b["id"]] = random.random() < b["reliability"]
                    success = all(individual_outcomes.values())
        elif isinstance(action, (int, np.integer)) and int(action) in bids:
            b = bids[int(action)]
            price, eta = b["price"], b["eta"]
            if (self._eligible(b, req) and price <= req["budget"]
                    and eta <= req["deadline"]):
                selected = [b["id"]]
                awarded = True
                observed_quality = b["quality"]
                success = random.random() < b["reliability"]
                individual_outcomes[b["id"]] = success
        if awarded:
            for i in selected:
                self.contractors[i]["capacity"] -= req["qty"]
                self.contractors[i]["workload"] += 1
            on_time = success and (random.random() < 0.90)
            observed_quality = observed_quality if success else 0.0
            reward = req["priority"] * (14.0 if success else -9.0) - 0.12 * price - 0.7 * eta
        else:
            # Rejected / no feasible contractor; no inventory is deducted.
            reward = -6.0 * req["priority"]
            observed_quality = 0.0
        feedback = {"awarded": awarded, "selected": selected,
                    "individual_outcomes": individual_outcomes, "success": success,
                    "on_time": on_time, "observed_quality": observed_quality,
                    "price": price if awarded else 0.0, "eta": eta if awarded else 0.0,
                    "request_id": req["id"]}
        self.history.append({"request": req["id"], "chosen": action,
                             "reward": float(reward), **feedback})
        self.pointer += 1
        done = self.pointer >= len(REQUESTS)
        return self._state(), float(reward), done, feedback


In [ ]:
# Cell 4: [TASK 1] Utility-based CNP manager and award selection
class BiddingManager:
    def __init__(self, num_contractors, alpha=ALPHA, gamma=GAMMA, epsilon=EPSILON):
        self.alpha, self.gamma, self.epsilon = alpha, gamma, epsilon
        self.trust = np.array([c["reliability"] for c in CONTRACTORS], dtype=float)
        self.workload = np.zeros(num_contractors, dtype=int)

    def choose_action(self, state):
        """Return a feasible best-value contractor (or picker-packer pair); -1 if none."""
        ### YOUR CODE HERE ###
        # TASK 1: check stock, budget, deadline, scenario metric and special capability.
        # Compute the utility shown in Cell 1. Use epsilon exploration and stable tie breaks.
        raise NotImplementedError("Complete Cell 4: choose_action")


In [ ]:
# Cell 5: [TASK 2] Update contract reliability estimates and workload
# This is a free-standing function so that Cell 5 remains a separately executable
# notebook cell. It is called from the locked Cell 6 simulation driver.
def update_agent(manager, state, action, feedback):
    """Update learned trust T[i] <- (1-alpha) T[i] + alpha * observed_signal."""
    ### YOUR CODE HERE ###
    # TASK 2: skip non-awards; calculate this problem's feedback signal.
    # Update trust (keep in [0, 1]) and per-episode workload for every winner.
    raise NotImplementedError("Complete Cell 5: update_agent")


In [ ]:
# Cell 6: Pre-built simulation runner (LOCKED)
env = EnvironmentEngine()
manager = BiddingManager(len(CONTRACTORS), alpha=ALPHA, gamma=GAMMA, epsilon=EPSILON)
rewards_history = []
steps_history = []
awards_history = []
success_history = []

for episode in range(NUM_EPISODES):
    state = env.reset()  # Reset contractor resources; retained trust learns across episodes.
    manager.workload[:] = 0  # Within-episode workload fairness starts afresh.
    cumulative_reward, awards, successes, steps = 0.0, 0, 0, 0
    done = False
    while not done and steps < MAX_STEPS_PER_EPISODE:
        action = manager.choose_action(state)
        next_state, reward, done, feedback = env.step(action)
        update_agent(manager, state, action, feedback)
        cumulative_reward += reward
        awards += int(feedback["awarded"])
        successes += int(feedback["success"])
        steps += 1
        state = next_state
    rewards_history.append(round(cumulative_reward, 3))
    steps_history.append(steps)
    awards_history.append(awards)
    success_history.append(successes)

print("Problem:", PROBLEM_ID, "|", PROBLEM_TITLE)
print("Completed episodes:", len(rewards_history))
print("Average reward (last 10 episodes):", round(float(np.mean(rewards_history[-10:])), 2))
print("Average contracts awarded (last 10):", round(float(np.mean(awards_history[-10:])), 2))
print("Average successful contracts (last 10):", round(float(np.mean(success_history[-10:])), 2))
print("Final learned contractor trust:", np.round(manager.trust, 3).tolist())
assert len(rewards_history) == NUM_EPISODES
assert all(0 <= x <= len(REQUESTS) for x in awards_history)
assert all(0 <= x <= len(REQUESTS) for x in success_history)
assert np.all((manager.trust >= 0) & (manager.trust <= 1)), "Trust must stay within [0, 1]"


In [ ]:
# Cell 7: Reward and contract-success visualizations (LOCKED)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(rewards_history, label="Reward / episode", linewidth=1.3)
ax[0].plot(np.convolve(rewards_history, np.ones(5) / 5, mode="valid"),
           label="5-episode moving average", linewidth=2)
ax[0].set(xlabel="Episode", ylabel="Total reward", title="Manager learning trajectory")
ax[0].legend(fontsize=8)
ax[0].grid(alpha=.3)
ax[1].plot(awards_history, label="Contracts awarded", alpha=.8)
ax[1].plot(success_history, label="Successful contracts", alpha=.8)
ax[1].set(xlabel="Episode", ylabel="Contracts", title="Allocation performance")
ax[1].set_ylim(0, len(REQUESTS) + 1)
ax[1].legend(fontsize=8)
ax[1].grid(alpha=.3)
fig.suptitle(PROBLEM_ID + ": " + PROBLEM_TITLE, fontsize=10)
plt.tight_layout()
plt.show()
